# NEON LiDAR CHM vs Meta/WRI v1 — 7주차 작은 비교 실험

2026-11-13 · 미실행 Colab 노트북임. CPU로 충분하며 GPU·NEON 토큰은 필요하지 않음.

Meta v1 높이 지도와 2021 NEON LiDAR를 23개 조사구에서 비교함.
6주차 면적 회귀계수를 고정한 채 높이 입력 교체가 AGB 예측에 미치는 영향도 확인함.
새로 읽는 외부 자료는 지정된 Meta **v1**의 작은 COG 창 및 선택적 관측일 메타데이터뿐임.
원본·기존 결과는 읽기만 하며 commit·push는 수행하지 않음.

6주차 features.csv(23행)와 coefficients.csv(2행)를 원문과 SHA-256으로 포함했음.
그 CSV를 따로 Drive로 옮길 필요는 없음. NEON 재현 검사 실패 시 Meta 읽기 전에 중단함.
조사구 특징은 각 자료 자체의 유효 픽셀로, 보조 픽셀 비교는 양쪽 모두 유효한 쌍으로 계산함.
bilinear·30m 버퍼·2m 피복 임계값을 고정했으며 재학습·보정·설정 탐색은 하지 않음.


## 1. 설치함
분석 패키지만 설치함. 원본 자료를 다운로드하는 설치 셀은 없음.


In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<4" "rasterio>=1.4,<2" "geopandas>=1.0,<2" "pyogrio>=0.9,<1" "matplotlib>=3.8,<4" "requests>=2.31,<3" "tabulate>=0.9,<1"


## 2. Drive를 마운트하고 출력 경로를 확인함
기존 출력 폴더 또는 ZIP이 있으면 덮어쓰지 않고 중단함. 분석 중간 배열은 메모리에만 둠.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
from contextlib import ExitStack
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse
from pyproj import CRS
import hashlib, io, json, math, re, shutil, tempfile, threading, time, zipfile
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import rasterio
from rasterio.features import geometry_mask, geometry_window
from rasterio.windows import Window
from rasterio.io import MemoryFile
from rasterio.merge import merge
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.plot import plotting_extent
from shapely.geometry import box, mapping, shape
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display, Markdown

ROOT = Path('/content/drive/MyDrive/neon_grsm_ornl')
RUN = ROOT / 'outputs/f55cc4636e94'
FOOTPRINT_PATH = RUN / 'sampled_footprints.gpkg'
REFERENCE_PATH = RUN / 'methods/A_DF_nested_OOF/plot_comparison.csv'
DRIVE_OUT = ROOT / 'outputs/meta_vs_neon_20261113'
ZIP_PATH = Path('/content/meta_vs_neon_results.zip')
META_BUCKET = 'https://dataforgood-fb-data.s3.amazonaws.com'
META_PREFIX = 'forests/v1/alsgedi_global_v6_float/'
BUFFER_M = 30.0

def ensure_new_outputs():
    for path in [DRIVE_OUT, ZIP_PATH]:
        if path.exists():
            raise FileExistsError(f'기존 결과 보호로 중단함: {path}. 이전 결과를 다른 위치에 보관한 뒤 재실행해야 함.')

ensure_new_outputs()
WORK = Path(tempfile.mkdtemp(prefix='meta_neon_', dir='/content'))
FIGURES = WORK / 'figures'
FIGURES.mkdir()
START_TIME = time.monotonic()
print('읽기 전용 입력:', ROOT)
print('이번 출력:', DRIVE_OUT, '| ZIP:', ZIP_PATH)


## 3. 입력 23개와 6주차 결과를 검증함
6주차 원문 CSV를 아래에 포함했음. 조사구 ID·지역 수·계수 방향·CHM 존재·좌표계를 검사함.
좌표 없는 나무도 포함한 AGB_ref_t_ha를 기준으로 사용하며 단위는 Mg/ha임.


In [ ]:
EXPECTED_IDS = ['GRSM_001', 'GRSM_009', 'GRSM_012', 'GRSM_014', 'GRSM_016', 'GRSM_017', 'GRSM_047', 'GRSM_049', 'GRSM_050', 'GRSM_051', 'GRSM_055', 'GRSM_057', 'GRSM_058', 'GRSM_059', 'GRSM_060', 'ORNL_001', 'ORNL_004', 'ORNL_006', 'ORNL_009', 'ORNL_010', 'ORNL_027', 'ORNL_049', 'ORNL_057']
EXPECTED_COUNTS = {'GRSM': 15, 'ORNL': 8}
FEATURES_CSV = 'plotID,siteID,polygon_area_m2,n_pixels,n_nodata,nodata_frac,n_zero,mean_height_m,cover_2m,AGB_ref_Mg_ha,AGB_A_Mg_ha,n_valid_pixels,nodata_warning_gt10pct\nGRSM_001,GRSM,388.4662149993231,391,0,0.0,0,30.50604084141724,1.0,232.82671062490672,185.0799779955425,391,False\nGRSM_009,GRSM,394.8012606836068,400,0,0.0,2,31.31681499004364,0.995,264.72228323097966,210.06796816537084,400,False\nGRSM_012,GRSM,382.24795822333203,383,0,0.0,0,29.982247983195453,1.0,827.6766620254415,242.29074521301857,383,False\nGRSM_014,GRSM,389.60795666454345,390,0,0.0,0,38.06342556293194,1.0,378.943446995208,336.9921428888424,390,False\nGRSM_016,GRSM,390.7502207868687,390,0,0.0,5,19.623866655887703,0.9871794871794872,331.62317653887527,88.31794481058303,390,False\nGRSM_017,GRSM,392.4654477436146,390,0,0.0,0,25.76632821498773,1.0,262.80339342702,161.8438109804341,390,False\nGRSM_047,GRSM,800.0,800,0,0.0,18,9.462356241568923,0.93125,92.50570348353776,26.72134320538139,800,False\nGRSM_049,GRSM,790.7495177800033,800,0,0.0,0,22.95897750414908,0.995,257.03563438958287,103.82627762364734,800,False\nGRSM_050,GRSM,797.7953916486747,800,0,0.0,0,31.508743716180323,1.0,287.1813250484601,312.7885268249207,800,False\nGRSM_051,GRSM,780.8522794556095,781,0,0.0,10,4.615720863607873,0.7720870678617158,17.685532007420477,21.98134433422115,781,False\nGRSM_055,GRSM,800.0,800,0,0.0,5,31.680370040684938,0.98875,466.6015190546865,343.2911193044526,800,False\nGRSM_057,GRSM,800.0,800,0,0.0,0,35.084507524967194,1.0,362.5993668720159,549.8880534278528,800,False\nGRSM_058,GRSM,799.5829044417083,800,0,0.0,0,34.32065630316734,1.0,271.1530181294086,300.5598276653628,800,False\nGRSM_059,GRSM,788.5626198200143,781,0,0.0,0,13.61283482578744,0.9871959026888605,141.5291863157999,51.24667062635993,781,False\nGRSM_060,GRSM,800.0,800,0,0.0,2,25.701718739271165,0.99,212.848723887494,180.414785659368,800,False\nORNL_001,ORNL,400.0000000122236,400,0,0.0,0,23.438444938659668,1.0,243.36343388895213,135.4635640187205,400,False\nORNL_004,ORNL,381.942356790429,382,0,0.0,0,28.446227777690787,1.0,244.94520316519103,58.16757044676908,382,False\nORNL_006,ORNL,390.32616855627003,390,0,0.0,0,29.461041069030763,1.0,182.2399243390523,208.2562136247944,390,False\nORNL_009,ORNL,400.0000000069849,400,0,0.0,0,23.1860674905777,1.0,294.6946805573866,168.61221489457176,400,False\nORNL_010,ORNL,393.49801743780193,391,0,0.0,0,21.1928081731967,1.0,231.76407020796955,143.94321108596617,391,False\nORNL_027,ORNL,390.2694661514088,391,0,0.0,0,14.44791816262638,1.0,102.48508424685548,40.040137888289365,391,False\nORNL_049,ORNL,799.9999999976717,800,0,0.0,0,23.427909989356994,1.0,147.17126747257214,139.43984051424974,800,False\nORNL_057,ORNL,800.000000015134,800,0,0.0,0,27.962816245555878,1.0,332.6348689933409,176.2675516316835,800,False\n'
FEATURES_ORIGINAL_SHA256 = '41ff333270d5b5acf0e086106cf3080f8b471d6434daaebf0a97310d8a714c41'
FEATURES_TEXT_SHA256 = '41ff333270d5b5acf0e086106cf3080f8b471d6434daaebf0a97310d8a714c41'
COEFFICIENTS_CSV = 'direction,train_site,test_site,b0,b1,b2,train_n,train_R2\nD1_GRSM_to_ORNL,GRSM,ORNL,-385.5655652750414,9.11890418658884,456.60815051757095,15,0.3593900956719329\nD2_ORNL_to_GRSM,ORNL,GRSM,18.746800248381135,8.505411493461185,0.0,8,0.29918662207626856\n'
COEFFICIENTS_ORIGINAL_SHA256 = '0a5746926b1a90cc35c7bf2af09af9d93e1f4d2ea679c8d2548823b817fc3b28'
COEFFICIENTS_TEXT_SHA256 = '0a5746926b1a90cc35c7bf2af09af9d93e1f4d2ea679c8d2548823b817fc3b28'

def require_columns(frame, columns, label):
    assert set(columns) <= set(frame.columns), f'{label}: 필수 열 누락 {set(columns) - set(frame.columns)}'

def assert_cohort(frame, label):
    require_columns(frame, ['plotID', 'siteID'], label)
    assert len(frame) == 23 and frame.plotID.is_unique, f'{label}: 고유 23행 필요함'
    assert set(frame.plotID) == set(EXPECTED_IDS), f'{label}: 6주차 ID와 다름'
    assert frame.siteID.value_counts().to_dict() == EXPECTED_COUNTS, f'{label}: GRSM 15 + ORNL 8 필요함'
    assert (frame.plotID.str.split('_').str[0] == frame.siteID).all()

assert hashlib.sha256(FEATURES_CSV.encode()).hexdigest() == FEATURES_TEXT_SHA256
assert hashlib.sha256(COEFFICIENTS_CSV.encode()).hexdigest() == COEFFICIENTS_TEXT_SHA256
week6_features = pd.read_csv(io.StringIO(FEATURES_CSV))
coefficients = pd.read_csv(io.StringIO(COEFFICIENTS_CSV))
assert_cohort(week6_features, '6주차 특징')
require_columns(week6_features, ['mean_height_m', 'cover_2m', 'AGB_ref_Mg_ha'], '6주차 특징')
require_columns(coefficients, ['direction', 'train_site', 'test_site', 'b0', 'b1', 'b2'], '6주차 계수')
assert len(coefficients) == 2 and coefficients.direction.is_unique
expected_directions = {'D1_GRSM_to_ORNL': ('GRSM', 'ORNL'), 'D2_ORNL_to_GRSM': ('ORNL', 'GRSM')}
assert set(coefficients.direction) == set(expected_directions)
for row in coefficients.itertuples():
    assert (row.train_site, row.test_site) == expected_directions[row.direction]
assert np.isfinite(coefficients[['b0', 'b1', 'b2']]).all().all()

CHM_PATHS = {pid: ROOT / 'data/crops' / f'{pid}_2021_chm_f55cc4636e94.tif' for pid in EXPECTED_IDS}
for path in [FOOTPRINT_PATH, REFERENCE_PATH, *CHM_PATHS.values()]:
    assert path.is_file(), f'기존 입력 파일 없음: {path}. 새 다운로드 없이 중단함.'
reference = pd.read_csv(REFERENCE_PATH, dtype={'plotID': str, 'siteID': str})
require_columns(reference, ['AGB_ref_t_ha'], '기준값')
assert_cohort(reference, '기준값')
assert np.isfinite(reference.AGB_ref_t_ha).all()
for col in ['field_year', 'imagery_year']:
    if col in reference:
        assert (reference[col] == 2021).all()
if 'C_ref_tC_ha' in reference:
    print('확인만 함: AGB × 0.5 = C_ref:', np.allclose(reference.AGB_ref_t_ha * 0.5, reference.C_ref_tC_ha))

layers = gpd.list_layers(FOOTPRINT_PATH)
if len(layers) == 1:
    layer = layers.iloc[0]['name']
else:
    display(layers)
    layer = input('조사구 경계 레이어 이름을 입력함: ').strip()
    assert layer in set(layers['name'])
footprints = gpd.read_file(FOOTPRINT_PATH, layer=layer)
assert footprints.crs is not None
assert len(footprints) == 23 and footprints.plotID.is_unique
assert set(footprints.plotID) == set(EXPECTED_IDS)
assert footprints.geometry.notna().all() and (~footprints.geometry.is_empty).all()
assert footprints.geometry.is_valid.all()
assert footprints.geom_type.isin(['Polygon', 'MultiPolygon']).all()
if 'siteID' in footprints:
    assert_cohort(footprints, '경계')
input_audit = []
for pid, path in CHM_PATHS.items():
    with rasterio.open(path) as src:
        assert src.crs and src.crs.to_epsg() in (32616, 32617), f'{pid}: CHM CRS 확인 필요함'
        assert src.count == 1 and np.allclose(src.res, (1, 1), rtol=0, atol=1e-6)
        assert abs(src.transform.b) < 1e-10 and abs(src.transform.d) < 1e-10
        input_audit.append(dict(plotID=pid, crs=str(src.crs), res=list(src.res), dtype=src.dtypes[0], nodata=src.nodata))
print('경계 원본 CRS:', footprints.crs)
display(pd.DataFrame(input_audit))
display(coefficients)


## 4. NEON 특징을 재계산하고 6주차 재현을 확인함
6주차의 float32 읽기·픽셀 중심 포함·0m와 결측 구분 규칙을 재사용했음.
mean_height_m, cover_2m이 저장값과 절대오차 1e-6 이내여야 진행함(rtol=0).
p95는 유효 픽셀에 numpy의 linear 백분위수로 추가 계산함. 결측률 >10%는 경고만 남김.


In [ ]:
GRIDS = {}

def neon_grid(pid):
    with rasterio.open(CHM_PATHS[pid]) as src:
        geom = footprints.loc[footprints.plotID == pid].to_crs(src.crs).geometry.iloc[0]
        window = geometry_window(src, [mapping(geom)], boundless=True)
        fill_nodata = src.nodata if src.nodata is not None else np.nan
        shape_ = (int(window.height), int(window.width))
        heights = np.full(shape_, fill_nodata, dtype='float32')
        source_valid = np.zeros(shape_, dtype=bool)
        c0, r0 = max(0, int(window.col_off)), max(0, int(window.row_off))
        c1 = min(src.width, int(window.col_off + window.width))
        r1 = min(src.height, int(window.row_off + window.height))
        if c1 > c0 and r1 > r0:
            read_window = Window(c0, r0, c1 - c0, r1 - r0)
            rs = slice(r0 - int(window.row_off), r1 - int(window.row_off))
            cs = slice(c0 - int(window.col_off), c1 - int(window.col_off))
            heights[rs, cs] = src.read(1, window=read_window, masked=False, out_dtype='float32')
            source_valid[rs, cs] = src.read_masks(1, window=read_window) > 0
        transform = src.window_transform(window)
        inside = geometry_mask([mapping(geom)], out_shape=heights.shape, transform=transform,
                               all_touched=False, invert=True)
        valid = inside & source_valid & np.isfinite(heights)
        if src.nodata is not None and np.isfinite(src.nodata):
            valid &= heights != np.float32(src.nodata)
        heights[~valid] = np.nan
        # 버퍼도 NEON 원래 격자의 정수 window로 정의함. 격자 원점을 새로 만들지 않음.
        buffered = geom.buffer(BUFFER_M)
        buffer_window = geometry_window(src, [mapping(buffered)], boundless=True)
        return dict(neon=heights, inside=inside, valid=valid, geom=geom, crs=src.crs,
                    transform=transform, window=window, polygon_area_m2=float(geom.area),
                    buffer_geom=buffered, buffer_window=buffer_window,
                    buffer_transform=src.window_transform(buffer_window))

def summarize(array, inside, pid):
    valid = inside & np.isfinite(array)
    n_pixels, n_valid = int(inside.sum()), int(valid.sum())
    assert n_pixels > 0 and n_valid > 0, f'{pid}: 유효 픽셀 0개이므로 전체 중단함'
    values = array[valid]
    n_nodata = n_pixels - n_valid
    return dict(n_pixels=n_pixels, n_valid_pixels=n_valid, n_nodata=n_nodata,
                nodata_frac=n_nodata / n_pixels, n_zero=int(np.count_nonzero(values == 0)),
                mean_height_m=float(values.mean(dtype=np.float64)),
                cover_2m=float(np.count_nonzero(values >= 2) / n_valid),
                p95_height_m=float(np.percentile(values, 95, method='linear')),
                nodata_warning_gt10pct=n_nodata / n_pixels > 0.10,
                n_negative=int(np.count_nonzero(values < 0)))

def extract_features(pid):
    grid = neon_grid(pid)
    GRIDS[pid] = grid
    return dict(plotID=pid, polygon_area_m2=grid['polygon_area_m2'],
                **summarize(grid['neon'], grid['inside'], pid))

neon_features = pd.DataFrame([extract_features(pid) for pid in EXPECTED_IDS])
neon_features = neon_features.merge(reference[['plotID', 'siteID', 'AGB_ref_t_ha']], on='plotID', validate='one_to_one')
assert_cohort(neon_features, 'NEON 재계산 특징')
check = neon_features.merge(week6_features, on='plotID', suffixes=('_new', '_week6'), validate='one_to_one')
for col in ['mean_height_m', 'cover_2m']:
    delta = (check[col + '_new'] - check[col + '_week6']).abs()
    print(col, '최대 절대 차이:', float(delta.max()))
    assert np.allclose(check[col + '_new'], check[col + '_week6'], rtol=0, atol=1e-6), f'{col}: 재현 실패로 중단함'
assert np.allclose(check.AGB_ref_t_ha, check.AGB_ref_Mg_ha, rtol=0, atol=1e-6), '기준 AGB가 6주차와 다름'
display(neon_features)


## 5. 필요한 zoom9 quadkey만 계산하고 관측일 메타데이터를 찾아봄
금정산 코드의 quadkey 함수 3개를 그대로 재사용했음. 조사구별 실제 30m 버퍼와 교차하는 타일만 남김.
관측일은 관측·영상 획득을 명시한 필드만 인정함. 파일 생성·수정일은 관측일로 쓰지 않음.
메타데이터가 없거나 크기 제한을 넘으면 미확인으로 기록하고 계속함.


In [ ]:
def quadkey_xy(x,y,z=9):
    return ''.join(str((1 if x&(1<<i) else 0)+(2 if y&(1<<i) else 0)) for i in range(z-1,-1,-1))

def quadkey_bounds(quad):
    x=y=0;z=len(quad)
    for c in quad:
        digit=int(c)
        if digit not in (0,1,2,3):raise ValueError('invalid quadkey')
        x=x*2+(digit&1);y=y*2+(digit>>1)
    def latitude(v):return math.degrees(math.atan(math.sinh(math.pi*(1-2*v/(2**z)))))
    return x/(2**z)*360-180,latitude(y+1),(x+1)/(2**z)*360-180,latitude(y)

def quadkeys_for_bounds(bounds,z=9):
    west,south,east,north=bounds
    def xy(lon,lat):
        lat=max(-85.05112878,min(85.05112878,lat)); n=2**z
        return max(0,min(n-1,int((lon+180)/360*n))),max(0,min(n-1,int((1-math.asinh(math.tan(math.radians(lat)))/math.pi)/2*n)))
    x0,y0=xy(west,north); x1,y1=xy(east,south)
    return [quadkey_xy(x,y,z) for y in range(y0,y1+1) for x in range(x0,x1+1)]
PLOT_QUADS, QUAD_URLS, META_RECORDS = {}, {}, {}
for pid in EXPECTED_IDS:
    region = gpd.GeoSeries([GRIDS[pid]['buffer_geom']], crs=GRIDS[pid]['crs']).to_crs(4326).iloc[0]
    quads = [q for q in quadkeys_for_bounds(region.bounds, 9)
             if region.intersection(box(*quadkey_bounds(q))).area > 0]
    assert quads, f'{pid}: quadkey 없음'
    PLOT_QUADS[pid] = quads
    GRIDS[pid]['region_ll'] = region
    for q in quads:
        QUAD_URLS[q] = META_BUCKET + '/' + META_PREFIX + 'chm/' + q + '.tif'
for q, url in sorted(QUAD_URLS.items()):
    META_RECORDS[q] = dict(quadkey=q, url=url, windows=[], warnings=[], http_requests=[])
print('읽을 v1 quadkey:', sorted(QUAD_URLS))
display(pd.DataFrame([dict(plotID=p, quadkeys=';'.join(PLOT_QUADS[p])) for p in EXPECTED_IDS]))

METADATA_AUDIT, METADATA_OBJECTS = [], []
MAX_METADATA_BYTES = 8 * 1024 * 1024  # 과학 설정이 아닌 작은 메타데이터 통신 한도임.

def small_get(url, params=None):
    with requests.get(url, params=params, stream=True, timeout=(15, 45)) as response:
        response.raise_for_status()
        length = response.headers.get('Content-Length')
        if length and int(length) > MAX_METADATA_BYTES:
            raise ValueError('선택적 메타데이터 크기 한도 초과함')
        parts, size = [], 0
        for part in response.iter_content(65536):
            size += len(part)
            if size > MAX_METADATA_BYTES:
                raise ValueError('선택적 메타데이터 크기 한도 초과함')
            parts.append(part)
        return b''.join(parts)

metadata_urls = {META_BUCKET + '/' + META_PREFIX + x for x in ['tiles.geojson', 'chm/tiles.geojson']}
try:
    listing = ET.fromstring(small_get(META_BUCKET, params={
        'list-type': '2', 'prefix': META_PREFIX, 'delimiter': '/', 'max-keys': 100}))
    for item in listing.findall('.//{*}Contents/{*}Key'):
        key = item.text or ''
        if key.endswith(('.geojson', '.json')):
            metadata_urls.add(META_BUCKET + '/' + key)
    date_prefixes = [e.text for e in listing.findall('.//{*}CommonPrefixes/{*}Prefix')
                     if any(t in (e.text or '').removeprefix(META_PREFIX).lower()
                            for t in ['date', 'metadata', 'footprint', 'geojson'])]
    for prefix in date_prefixes:
        for q in QUAD_URLS:
            try:
                tile_listing = ET.fromstring(small_get(META_BUCKET, params={
                    'list-type': '2', 'prefix': prefix + q, 'max-keys': 20}))
                for e in tile_listing.findall('.//{*}Contents/{*}Key'):
                    if (e.text or '').endswith(('.geojson', '.json')):
                        metadata_urls.add(META_BUCKET + '/' + e.text)
            except Exception as exc:
                METADATA_AUDIT.append(dict(url=prefix + q, status=str(exc)))
except Exception as exc:
    METADATA_AUDIT.append(dict(url=META_BUCKET + '/' + META_PREFIX, status=str(exc)))
for url in sorted(metadata_urls):
    try:
        obj = json.loads(small_get(url))
        METADATA_OBJECTS.append((url, obj))
        METADATA_AUDIT.append(dict(url=url, status='read'))
    except Exception as exc:
        METADATA_AUDIT.append(dict(url=url, status='미확인: ' + str(exc)))

def observation_fields(properties):
    found = []
    for key, value in properties.items():
        k = re.sub(r'[^a-z0-9]', '', str(key).lower())
        explicit = any(t in k for t in ['acquis', 'observ', 'capture', 'sensing', 'image', 'imagery'])
        date_like = any(t in k for t in ['date', 'time', 'year'])
        if explicit and date_like and value is not None and not isinstance(value, (dict, list)):
            found.append(f'{key}={value}')
    return found

def metadata_dates(pid):
    dates = []
    region = GRIDS[pid]['region_ll']
    for url, obj in METADATA_OBJECTS:
        for feature in obj.get('features', []) if isinstance(obj, dict) else []:
            props = feature.get('properties') or {}
            try:
                if feature.get('geometry'):
                    geom = gpd.GeoSeries([shape(feature['geometry'])], crs=(obj.get('crs') or {}).get('properties', {}).get('name', 'EPSG:4326')).to_crs(4326).iloc[0]
                    if not geom.intersects(region):
                        continue
                elif not any(q in json.dumps(props) for q in PLOT_QUADS[pid]):
                    continue
                dates.extend(url + ' | ' + item for item in observation_fields(props))
            except Exception as exc:
                METADATA_AUDIT.append(dict(url=url, plotID=pid, status='날짜 공간연결 실패: ' + str(exc)))
        if isinstance(obj, dict) and obj.get('type') != 'FeatureCollection' and any(q in url for q in PLOT_QUADS[pid]):
            dates.extend(url + ' | ' + item for item in observation_fields(obj))
    return dates
print('선택적 메타데이터 확인 결과:', METADATA_AUDIT)


## 6. HTTP Range를 강제해 Meta COG의 작은 창만 읽음
금정산의 원격 rasterio window 읽기를 재사용함. 전체 TIFF 다운로드로 전환하지 않음.
통신 중계는 Range 없는 GET과 206이 아닌 응답을 거절하며 전송량을 기록함.
strip TIFF도 허용함. 23개 조사구의 블록 읽기량을 먼저 계산하며 조사구당 300 MB 초과 시 중단함.
300 MB는 압축 전 블록 바이트 합계 기준이며, 실제 네트워크 전송량과 구분함.
타일 전체를 포함하는 window는 거절함. 실패 시 이유를 출력하고 전체 중단함.
해상도·CRS·dtype·nodata·tags·unit을 저장함. 단위 m이 불확실하면 경고만 남김.


In [ ]:
TRANSFER = {'bytes': 0, 'errors': []}
TRANSFER_LOCK = threading.Lock()

def range_fetch(q, start, end):
    url = QUAD_URLS[q]
    with requests.get(url, headers={'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity'},
                      stream=True, timeout=(15, 90)) as response:
        match = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)', response.headers.get('Content-Range', ''))
        if response.status_code != 206 or not match:
            raise RuntimeError(f'HTTP Range 실패: {q}, status={response.status_code}, Content-Range={response.headers.get("Content-Range")}. 전체 다운로드 없이 중단함.')
        a, b, total = map(int, match.groups())
        if a != start or b != min(end, total - 1) or b - a + 1 >= total:
            raise RuntimeError(f'잘못된 Range/전체 파일 응답: {q}')
        parts, received = [], 0
        for part in response.iter_content(65536):
            received += len(part)
            with TRANSFER_LOCK:
                TRANSFER['bytes'] += len(part)
            if received > b - a + 1:
                raise RuntimeError('요청 범위보다 큰 응답이므로 중단함')
            parts.append(part)
        if received != b - a + 1:
            raise RuntimeError('Range 응답 길이 불일치함')
        with TRANSFER_LOCK:
            META_RECORDS[q]['http_requests'].append(dict(start=a, end=b, bytes=received))
        return b''.join(parts), total

for q in QUAD_URLS:
    try:
        _, total = range_fetch(q, 0, 16383)
        META_RECORDS[q]['object_bytes'] = total
    except Exception as exc:
        print('Meta Range 사전 확인 실패함:', exc)
        raise

class RangeOnlyHandler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def do_HEAD(self):
        q = Path(urlparse(self.path).path).stem
        if q not in QUAD_URLS:
            self.send_error(404)
            return
        self.send_response(200)
        self.send_header('Content-Type', 'image/tiff')
        self.send_header('Content-Length', str(META_RECORDS[q]['object_bytes']))
        self.send_header('Accept-Ranges', 'bytes')
        self.end_headers()

    def do_GET(self):
        q = Path(urlparse(self.path).path).stem
        match = re.fullmatch(r'bytes=(\d+)-(\d+)', self.headers.get('Range', ''))
        if q not in QUAD_URLS or not match:
            self.send_error(416, 'Only explicit byte ranges are allowed')
            return
        a, b = map(int, match.groups())
        # 과학 설정이 아닌 개별 통신 요청의 크기 보호 한도임.
        if b < a or b - a + 1 > 32 * 1024 * 1024 or b - a + 1 >= META_RECORDS[q]['object_bytes']:
            self.send_error(416, 'Range too large; full-tile fallback is forbidden')
            return
        try:
            payload, total = range_fetch(q, a, b)
            self.send_response(206)
            self.send_header('Content-Type', 'image/tiff')
            self.send_header('Accept-Ranges', 'bytes')
            self.send_header('Content-Range', f'bytes {a}-{a + len(payload) - 1}/{total}')
            self.send_header('Content-Length', str(len(payload)))
            self.end_headers()
            self.wfile.write(payload)
        except Exception as exc:
            TRANSFER['errors'].append(str(exc))
            self.send_error(502, 'Remote window read failed; no full download fallback')

server = ThreadingHTTPServer(('127.0.0.1', 0), RangeOnlyHandler)
server.daemon_threads = True
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
NATIVE = {}
MAX_PLOT_READ_BYTES = 300_000_000  # 300 MB (십진 단위), 조사구의 모든 타일 window 합계임.

def estimate_window_read(src, window):
    block_h, block_w = src.block_shapes[0]
    block_rows = math.ceil((window.row_off + window.height) / block_h) - math.floor(window.row_off / block_h)
    block_cols = math.ceil((window.col_off + window.width) / block_w) - math.floor(window.col_off / block_w)
    n_blocks = int(block_rows * block_cols)
    max_bytes = int(n_blocks * block_h * block_w * np.dtype(src.dtypes[0]).itemsize)
    return n_blocks, max_bytes

try:
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif',
                      GDAL_HTTP_MULTIRANGE='NO', GDAL_HTTP_MAX_RETRY=5, GDAL_HTTP_RETRY_DELAY=2, VSI_CACHE=True):
        with ExitStack() as stack:
            opened = {q: stack.enter_context(rasterio.open(f'http://127.0.0.1:{server.server_port}/{q}.tif'))
                      for q in QUAD_URLS}
            for q, src in opened.items():
                assert src.crs is not None and src.count == 1, f'{q}: CHM CRS/밴드 확인 필요함'
                assert abs(src.transform.b) < 1e-10 and abs(src.transform.d) < 1e-10
                namespaces = src.tag_namespaces()
                tags = dict(default=src.tags(), band=src.tags(1),
                            namespaces={ns: src.tags(ns=ns) for ns in namespaces if ns})
                unit_fields = {k: v for group in [tags['default'], tags['band']] for k, v in group.items()
                               if 'unit' in k.lower()}
                unit_values = [str(x).strip().lower() for x in [src.units[0], *unit_fields.values()] if x]
                meters_confirmed = any(v in ['m', 'meter', 'meters', 'metre', 'metres'] for v in unit_values)
                unit_status = 'm confirmed in source metadata' if meters_confirmed else 'WARNING: height unit unconfirmed; assumed m without conversion'
                if not meters_confirmed:
                    META_RECORDS[q]['warnings'].append(unit_status)
                axis_units = [a.unit_name for a in CRS(src.crs).axis_info]
                META_RECORDS[q].update(crs=str(src.crs), resolution=list(src.res),
                    crs_axis_units=axis_units, dtype=src.dtypes[0],
                    nodata=str(src.nodata) if src.nodata is not None and not np.isfinite(src.nodata) else src.nodata,
                    width=src.width, height=src.height, is_tiled=bool(src.is_tiled),
                    block_shapes=src.block_shapes, compression=src.compression.name if src.compression else None,
                    tags=tags, band_unit=src.units[0], unit_fields=unit_fields, height_unit_status=unit_status)
                print(q, {k: META_RECORDS[q][k] for k in ['crs', 'resolution', 'crs_axis_units', 'dtype', 'nodata', 'height_unit_status',
                                                         'is_tiled', 'block_shapes', 'compression', 'width', 'height']})

            # 모든 조사구의 읽기량을 먼저 검사함. 이 단계에서는 픽셀을 읽지 않음.
            READ_PLANS, estimate_rows = {}, []
            for pid in EXPECTED_IDS:
                plans, grid = [], GRIDS[pid]
                for q in PLOT_QUADS[pid]:
                    src = opened[q]
                    region = gpd.GeoSeries([grid['buffer_geom']], crs=grid['crs']).to_crs(src.crs).iloc[0]
                    region = region.intersection(box(*src.bounds))
                    if region.is_empty:
                        continue
                    window = geometry_window(src, [mapping(region)], boundless=False)
                    assert window.width * window.height < src.width * src.height, '전체 타일 window를 거절함'
                    n_blocks, max_bytes = estimate_window_read(src, window)
                    record = dict(plotID=pid, col_off=int(window.col_off), row_off=int(window.row_off),
                        width=int(window.width), height=int(window.height), buffer_m=BUFFER_M,
                        n_blocks=n_blocks, estimated_uncompressed_block_bytes=max_bytes,
                        estimated_uncompressed_block_MB=max_bytes / 1_000_000)
                    plans.append(dict(quadkey=q, window=window, record=record))
                assert plans, f'{pid}: Meta window 없음. 조사구를 빼지 않고 전체 중단함'
                total_bytes = sum(p['record']['estimated_uncompressed_block_bytes'] for p in plans)
                for plan in plans:
                    plan['record'].update(plot_estimated_read_bytes=total_bytes,
                        plot_estimated_read_MB=total_bytes / 1_000_000,
                        plot_read_limit_MB=300, plot_read_within_limit=total_bytes <= MAX_PLOT_READ_BYTES)
                    META_RECORDS[plan['quadkey']]['windows'].append(plan['record'])
                READ_PLANS[pid] = plans
                estimate_rows.append(dict(plotID=pid, n_windows=len(plans),
                    estimated_read_MB=total_bytes / 1_000_000, within_300_MB=total_bytes <= MAX_PLOT_READ_BYTES))
            read_estimates = pd.DataFrame(estimate_rows)
            display(read_estimates)
            print('300 MB 이하 조사구:', int(read_estimates.within_300_MB.sum()), '/', len(EXPECTED_IDS))
            exceeded = read_estimates.loc[~read_estimates.within_300_MB]
            if not exceeded.empty:
                reason = '; '.join(f'{r.plotID}: {r.estimated_read_MB:.2f} MB' for r in exceeded.itertuples())
                raise RuntimeError('조사구당 압축 전 블록 읽기량 300 MB 초과함: ' + reason + '. 전체 다운로드 없이 중단함.')

            for pid in EXPECTED_IDS:
                windows = []
                for plan in READ_PLANS[pid]:
                    q, window = plan['quadkey'], plan['window']
                    src = opened[q]
                    raw = src.read(1, window=window, out_dtype='float32', masked=False)
                    valid = (src.read_masks(1, window=window) > 0) & np.isfinite(raw)
                    if src.nodata is not None and np.isfinite(src.nodata):
                        valid &= raw != np.float32(src.nodata)
                    raw[~valid] = np.nan  # 0m/음수 유효값은 임의 보정하지 않음.
                    windows.append(dict(quadkey=q, array=raw, transform=src.window_transform(window), crs=src.crs))
                assert windows, f'{pid}: Meta window 없음. 조사구를 빼지 않고 전체 중단함'
                NATIVE[pid] = windows
                print(pid, '작은 창 읽기 완료함 | 누적 Meta TIFF 전송 MiB:', round(TRANSFER['bytes'] / 2**20, 2))
except Exception as exc:
    print('Meta window 읽기 실패로 전체 중단함. 전체 다운로드로 전환하지 않음:', exc)
    print('통신 원인:', TRANSFER['errors'])
    raise
finally:
    server.shutdown()
    server.server_close()
print('실제 TIFF 전송 MiB (중복 Range·헤더 포함):', TRANSFER['bytes'] / 2**20)


## 7. Meta 창을 NEON의 CRS·1m 격자로 정렬함
인접 창은 원래 격자에서 합친 뒤 bilinear 한 번으로 재투영함. 원본 격자가 서로 다르면 중단함.
목적 격자는 30m 버퍼를 포함한 NEON 정수 window임. 이후 원래 조사구 window를 정확히 잘라 비교함.
결측은 NaN으로 명시함. 음수나 0m 유효값을 보정하지 않음.


In [ ]:
ALIGNED, PLOT_DATES = {}, {}
for pid in EXPECTED_IDS:
    grid, parts = GRIDS[pid], NATIVE[pid]
    first = parts[0]
    for part in parts:
        assert part['crs'] == first['crs']
        a, b = first['transform'], part['transform']
        assert np.allclose([a.a, a.e], [b.a, b.e], rtol=0, atol=1e-12)
        phase = [(b.c - a.c) / a.a, (b.f - a.f) / a.e]
        assert np.allclose(phase, np.round(phase), rtol=0, atol=1e-5), 'Meta 창 원본 격자 불일치함'
    with ExitStack() as stack:
        sources = []
        for part in parts:
            memory = stack.enter_context(MemoryFile())
            ds = stack.enter_context(memory.open(driver='GTiff', height=part['array'].shape[0],
                width=part['array'].shape[1], count=1, dtype='float32', crs=part['crs'],
                transform=part['transform'], nodata=np.nan))
            ds.write(part['array'], 1)
            sources.append(ds)
        native, transform = merge(sources, indexes=[1], dtype='float32', nodata=np.nan, method='first')
    bw = grid['buffer_window']
    target = np.full((int(bw.height), int(bw.width)), np.nan, dtype='float32')
    reproject(native[0], target, src_transform=transform, src_crs=first['crs'], src_nodata=np.nan,
              dst_transform=grid['buffer_transform'], dst_crs=grid['crs'], dst_nodata=np.nan,
              resampling=Resampling.bilinear, init_dest_nodata=True)
    pw = grid['window']
    row = int(pw.row_off - bw.row_off)
    col = int(pw.col_off - bw.col_off)
    assert row >= 0 and col >= 0
    aligned = target[row:row + grid['neon'].shape[0], col:col + grid['neon'].shape[1]].copy()
    assert aligned.shape == grid['neon'].shape
    aligned[~grid['inside']] = np.nan
    ALIGNED[pid] = aligned
    assert np.isfinite(aligned[grid['inside']]).any(), f'{pid}: Meta 유효 픽셀 0개로 전체 중단함'
    dates = metadata_dates(pid)
    for q in PLOT_QUADS[pid]:
        rec = META_RECORDS[q]
        for group in ['default', 'band']:
            dates.extend('tile tag ' + q + ' | ' + item for item in observation_fields(rec['tags'][group]))
    PLOT_DATES[pid] = '; '.join(sorted(set(dates))) or '미확인'
print('NEON 격자로 bilinear 정렬했음. Meta 관측일 미확인 조사구:', sum(v == '미확인' for v in PLOT_DATES.values()))


## 8. 조사구 특징을 비교함 (주 결과)
두 자료에서 동일한 경계·픽셀 중심 포함 규칙을 사용함. 각 자료 자체의 유효 픽셀로 요약함.
Meta−NEON의 편향·RMSE·MAE·r·평균비를 특징별, 사이트별, 통합 23개로 계산함.
cover_2m 오차는 0~1 비율 단위임. 결측률 >10%도 제외하지 않음.


In [ ]:
rows = []
for pid in EXPECTED_IDS:
    n = neon_features.loc[neon_features.plotID == pid].iloc[0]
    m = summarize(ALIGNED[pid], GRIDS[pid]['inside'], pid)
    row = dict(plotID=pid, siteID=n.siteID, polygon_area_m2=n.polygon_area_m2,
               AGB_ref_Mg_ha=n.AGB_ref_t_ha, meta_observation_date=PLOT_DATES[pid],
               meta_quadkeys=';'.join(PLOT_QUADS[pid]),
               meta_unit_warning=any(META_RECORDS[q]['warnings'] for q in PLOT_QUADS[pid]))
    for key in m:
        row['neon_' + key] = n[key]
        row['meta_' + key] = m[key]
    for key in ['mean_height_m', 'cover_2m', 'p95_height_m']:
        row['delta_' + key] = m[key] - n[key]
    rows.append(row)
plot_features = pd.DataFrame(rows)
assert_cohort(plot_features, '비교 특징')

def metrics(neon_or_ref, meta_or_pred):
    x, y = np.asarray(neon_or_ref, dtype=np.float64), np.asarray(meta_or_pred, dtype=np.float64)
    assert len(x) == len(y) and len(x) > 0
    assert np.isfinite(x).all() and np.isfinite(y).all(), '지표에서 조사구를 제외하지 않음'
    error = y - x
    return dict(n=len(x), bias=float(error.mean()), RMSE=float(np.sqrt(np.mean(error**2))),
                MAE=float(np.mean(np.abs(error))),
                pearson_r=float(np.corrcoef(x, y)[0, 1]) if np.std(x) > 0 and np.std(y) > 0 else np.nan,
                mean_ratio=float(y.mean() / x.mean()) if x.mean() != 0 else np.nan)

def scopes(frame):
    return [('GRSM', frame.loc[frame.siteID == 'GRSM']), ('ORNL', frame.loc[frame.siteID == 'ORNL']), ('ALL_23', frame)]

plot_metric_rows = []
for scope, q in scopes(plot_features):
    for feature in ['mean_height_m', 'cover_2m', 'p95_height_m']:
        plot_metric_rows.append(dict(scope=scope, feature=feature, unit='fraction' if feature == 'cover_2m' else 'm',
            **metrics(q['neon_' + feature], q['meta_' + feature])))
plot_metrics = pd.DataFrame(plot_metric_rows)
display(plot_features)
display(plot_metrics)


## 9. 유효 픽셀 쌍을 비교함 (보조 결과)
양쪽 모두 유효한 경계 내부 픽셀만 쌍으로 사용함. 한 조사구라도 쌍이 0개면 전체 중단함.
높이 구간은 NEON 기준 [0,10), [10,20), [20,30), [30,∞)m임.
음수 NEON 값은 전체 지표에 유지하고 별도 구간으로 기록함. 픽셀은 독립 표본으로 해석하지 않음.


In [ ]:
pixel_parts = []
for pid in EXPECTED_IDS:
    grid = GRIDS[pid]
    paired = grid['inside'] & np.isfinite(grid['neon']) & np.isfinite(ALIGNED[pid])
    assert paired.any(), f'{pid}: 유효 픽셀 쌍 0개로 전체 중단함'
    site = pid.split('_')[0]
    pixel_parts.append(pd.DataFrame(dict(plotID=pid, siteID=site,
        neon_height_m=grid['neon'][paired], meta_height_m=ALIGNED[pid][paired])))
pixels = pd.concat(pixel_parts, ignore_index=True)
pixel_metric_rows, bin_rows = [], []
HEIGHT_BINS = [('0-10', 0, 10), ('10-20', 10, 20), ('20-30', 20, 30), ('30+', 30, np.inf)]
for scope, q in scopes(pixels):
    stats = metrics(q.neon_height_m, q.meta_height_m)
    pixel_metric_rows.append(dict(scope=scope, n_plots=q.plotID.nunique(), n_pairs=stats['n'],
                                  bias_m=stats['bias'], RMSE_m=stats['RMSE'], pearson_r=stats['pearson_r']))
    bins = HEIGHT_BINS + ([('negative_NEON_audit', -np.inf, 0)] if (q.neon_height_m < 0).any() else [])
    for label, low, high in bins:
        selected = q.loc[(q.neon_height_m >= low) & (q.neon_height_m < high)]
        bin_rows.append(dict(scope=scope, neon_height_bin_m=label, n_pairs=len(selected),
            bias_m=float((selected.meta_height_m - selected.neon_height_m).mean()) if len(selected) else np.nan))
pixel_metrics = pd.DataFrame(pixel_metric_rows)
height_bin_bias = pd.DataFrame(bin_rows)
display(pixel_metrics)
display(height_bin_bias)


## 10. 6주차 계수를 그대로 적용해 AGB 영향을 확인함
GRSM은 D2, ORNL은 D1 계수만 사용함. 회귀 학습·변수 추가·보정은 하지 않음.
음수 AGB 예측도 그대로 유지하고 flag를 남김. 두 예측을 같은 기준 AGB와 비교함.


In [ ]:
plot_features['direction'] = plot_features.siteID.map({'GRSM': 'D2_ORNL_to_GRSM', 'ORNL': 'D1_GRSM_to_ORNL'})
for source in ['neon', 'meta']:
    predictions = []
    for row in plot_features.itertuples():
        coef = coefficients.loc[coefficients.direction == row.direction].iloc[0]
        assert coef.test_site == row.siteID and coef.train_site != row.siteID
        prediction = coef.b0 + coef.b1 * getattr(row, source + '_mean_height_m') + coef.b2 * getattr(row, source + '_cover_2m')
        predictions.append(float(prediction))
    plot_features['AGB_' + source + '_pred_Mg_ha'] = predictions
    plot_features['error_AGB_' + source + '_Mg_ha'] = plot_features['AGB_' + source + '_pred_Mg_ha'] - plot_features.AGB_ref_Mg_ha
    plot_features['negative_AGB_' + source + '_flag'] = plot_features['AGB_' + source + '_pred_Mg_ha'] < 0
plot_features['delta_AGB_meta_minus_neon_Mg_ha'] = plot_features.AGB_meta_pred_Mg_ha - plot_features.AGB_neon_pred_Mg_ha
agb_rows = []
for scope, q in scopes(plot_features):
    for source in ['neon', 'meta']:
        stats = metrics(q.AGB_ref_Mg_ha, q['AGB_' + source + '_pred_Mg_ha'])
        agb_rows.append(dict(scope=scope, input_source=source.upper(), n=stats['n'],
            RMSE_Mg_ha=stats['RMSE'], bias_Mg_ha=stats['bias'], mean_ratio=stats['mean_ratio'], pearson_r=stats['pearson_r']))
agb_comparison = pd.DataFrame(agb_rows)
display(agb_comparison)
print('계수 재학습 없이 두 입력을 비교했음. 음수 예측도 보존했음.')


## 11. 영어 라벨 그림 5개를 만듦
예시 지도는 조사구 평균 높이의 부호 있는 Meta−NEON 편향이 가장 작은 곳과 가장 큰 곳을 선택함.
동일한 높이 색 범위와 0 중심의 차이 색 범위를 사용함. 음수값과 결측도 숨기지 않음.


In [ ]:
COLORS = {'GRSM': 'tab:blue', 'ORNL': 'tab:orange'}

def diagonal(ax, x, y):
    values = np.concatenate([np.asarray(x).ravel(), np.asarray(y).ravel()])
    values = values[np.isfinite(values)]
    low, high = min(0.0, float(values.min())), max(1.0, float(values.max()))
    margin = (high - low) * 0.05
    limits = [low - margin, high + margin]
    ax.plot(limits, limits, 'k--', lw=1, label='1:1')
    ax.set(xlim=limits, ylim=limits)
    ax.set_aspect('equal', adjustable='box')

def finish(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 6))
for site, q in plot_features.groupby('siteID'):
    ax.scatter(q.neon_mean_height_m, q.meta_mean_height_m, color=COLORS[site], label=site)
diagonal(ax, plot_features.neon_mean_height_m, plot_features.meta_mean_height_m)
ax.set(xlabel='NEON mean height (m)', ylabel='Meta mean height (m)', title='Plot mean height (n=23)')
ax.legend(); ax.grid(alpha=0.2)
finish(fig, 'scatter_plot_mean_height.png')

fig, ax = plt.subplots(figsize=(6.5, 6))
hexbin = ax.hexbin(pixels.neon_height_m, pixels.meta_height_m, gridsize=60, mincnt=1, bins='log', cmap='viridis')
diagonal(ax, pixels.neon_height_m, pixels.meta_height_m)
ax.set(xlabel='NEON pixel height (m)', ylabel='Meta pixel height (m)', title=f'Valid paired pixels (n={len(pixels):,})')
fig.colorbar(hexbin, ax=ax, label='Pixel pair count (log scale)')
finish(fig, 'pixel_hexbin.png')

fig, ax = plt.subplots(figsize=(8, 4.5))
labels = [b[0] for b in HEIGHT_BINS]
positions = np.arange(4)
for offset, scope, color in [(-0.25, 'GRSM', COLORS['GRSM']), (0, 'ORNL', COLORS['ORNL']), (0.25, 'ALL_23', 'gray')]:
    q = height_bin_bias.loc[height_bin_bias.scope == scope].set_index('neon_height_bin_m').reindex(labels)
    ax.bar(positions + offset, q.bias_m, width=0.24, label=scope, color=color)
ax.axhline(0, color='black', lw=1)
ax.set(xticks=positions, xticklabels=labels, xlabel='NEON height bin (m)', ylabel='Meta - NEON bias (m)', title='Paired-pixel bias by reference height')
ax.legend()
finish(fig, 'bias_by_height_bin.png')

fig, ax = plt.subplots(figsize=(7, 6.5))
for site, q in plot_features.groupby('siteID'):
    ax.scatter(q.AGB_ref_Mg_ha, q.AGB_neon_pred_Mg_ha, color=COLORS[site], marker='o', label=f'{site}: NEON input')
    ax.scatter(q.AGB_ref_Mg_ha, q.AGB_meta_pred_Mg_ha, edgecolors=COLORS[site], facecolors='none', marker='^', label=f'{site}: Meta input')
diagonal(ax, plot_features.AGB_ref_Mg_ha, plot_features[['AGB_neon_pred_Mg_ha', 'AGB_meta_pred_Mg_ha']].to_numpy())
ax.set(xlabel='Reference AGB (Mg/ha)', ylabel='Predicted AGB (Mg/ha)', title='Frozen week-6 coefficients: input substitution')
ax.legend(fontsize=8); ax.grid(alpha=0.2)
finish(fig, 'agb_neon_vs_meta.png')

ranked = plot_features.sort_values(['delta_mean_height_m', 'plotID'])
example_ids = [ranked.iloc[0].plotID, ranked.iloc[-1].plotID]
assert len(set(example_ids)) == 2
heights = np.concatenate([np.concatenate([GRIDS[p]['neon'].ravel(), ALIGNED[p].ravel()]) for p in example_ids])
heights = heights[np.isfinite(heights)]
vmin, vmax = min(0.0, float(heights.min())), max(1.0, float(heights.max()))
diffs = [ALIGNED[p] - GRIDS[p]['neon'] for p in example_ids]
limit = max(1.0, max(float(np.nanmax(np.abs(d))) for d in diffs))
fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
for i, pid in enumerate(example_ids):
    grid = GRIDS[pid]
    extent = plotting_extent(grid['neon'], grid['transform'])
    for j, (array, title) in enumerate([(grid['neon'], 'NEON'), (ALIGNED[pid], 'Meta'), (diffs[i], 'Meta - NEON')]):
        ax = axes[i, j]
        image = ax.imshow(array, extent=extent, origin='upper', cmap='RdBu_r' if j == 2 else 'viridis',
                          norm=TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit) if j == 2 else plt.Normalize(vmin, vmax))
        gpd.GeoSeries([grid['geom']], crs=grid['crs']).boundary.plot(ax=ax, color='black', linewidth=0.7)
        ax.set(title=f'{pid}: {title}', xlabel='Easting (m)', ylabel='Northing (m)')
        ax.ticklabel_format(style='plain', useOffset=False)
        fig.colorbar(image, ax=ax, label='Height difference (m)' if j == 2 else 'Height (m)', shrink=0.75)
fig.suptitle('Examples: smallest and largest signed plot mean bias')
fig.savefig(FIGURES / 'examples.png', dpi=180, bbox_inches='tight')
plt.show(); plt.close(fig)


## 12. 표·출처·보고서를 새 폴더에 저장하고 ZIP을 다운로드함
기존 경로가 있으면 다시 중단함. ZIP과 Drive 사본의 해시가 같은지 확인함.
보고서에 시점·학습자료 겹침·내부 비교 한계를 명시하며 다음 단계 판단은 빈칸으로 둠.


In [ ]:
ensure_new_outputs()
tables = {'plot_features.csv': plot_features, 'plot_metrics.csv': plot_metrics,
          'pixel_metrics.csv': pixel_metrics, 'height_bin_bias.csv': height_bin_bias,
          'agb_comparison.csv': agb_comparison}
for filename, frame in tables.items():
    frame.to_csv(WORK / filename, index=False)

def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return int(value)
    return value

sources = dict(product='Meta/WRI Global Canopy Height v1', prefix=META_PREFIX,
    resampling='bilinear', buffer_m=BUFFER_M, cover_threshold_m=2,
    tiff_transfer_bytes=TRANSFER['bytes'], tiff_transfer_errors=TRANSFER['errors'],
    transfer_note='Actual TIFF HTTP bodies including repeated ranges and metadata probes; optional JSON/XML excluded',
    metadata_audit=METADATA_AUDIT, tiles=list(META_RECORDS.values()),
    plot_observation_dates=PLOT_DATES, coefficients_original_sha256=COEFFICIENTS_ORIGINAL_SHA256,
    features_original_sha256=FEATURES_ORIGINAL_SHA256,
    frozen_coefficients=coefficients.to_dict(orient='records'), neon_input_audit=input_audit,
    elapsed_seconds=time.monotonic() - START_TIME)
(WORK / 'meta_sources.json').write_text(json.dumps(clean_json(sources), ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')

all_mean = plot_metrics.loc[(plot_metrics.scope == 'ALL_23') & (plot_metrics.feature == 'mean_height_m')].iloc[0]
agb_all = agb_comparison.loc[agb_comparison.scope == 'ALL_23'].set_index('input_source')
headline = (f"내부 비교 23개: Meta−NEON 조사구 평균 높이 편향 {all_mean.bias:+.2f}m, RMSE {all_mean.RMSE:.2f}m; "
            f"고정 계수 AGB RMSE NEON {agb_all.loc['NEON', 'RMSE_Mg_ha']:.2f} → Meta {agb_all.loc['META', 'RMSE_Mg_ha']:.2f} Mg/ha임.")
report = headline + '\n\n# Meta v1 vs NEON CHM — 2026-11-13\n\n'
for title, frame in [('조사구 특징 비교 (주 결과)', plot_metrics), ('픽셀 쌍 비교 (보조 결과)', pixel_metrics),
                     ('NEON 높이 구간별 편향', height_bin_bias), ('고정 계수 AGB 비교', agb_comparison)]:
    report += '## ' + title + '\n\n' + frame.to_markdown(index=False, floatfmt='.4f') + '\n\n'
report += """편향은 Meta−NEON, AGB에서는 예측−기준으로 정의했음. 비율은 평균 추정÷평균 기준임.
조사구 특징은 각각의 유효 픽셀로 계산했으며 픽셀 비교는 양쪽 모두 유효한 쌍만 사용했음.
통합 픽셀 지표는 모든 쌍을 합친 픽셀 가중 결과이며, 조사구 지표는 조사구마다 같은 가중치임.
0m는 유효값으로 포함했음. 결측률 10% 초과도 경고만 남겼으며 조사구는 제외하지 않았음.
분모가 0인 비율과 상수 벡터의 r은 NaN으로 기록했음. 음수 높이·AGB는 임의 보정하지 않았음.
높이 구간은 NEON 기준 [0,10), [10,20), [20,30), [30,∞)m이며 음수가 있으면 별도 점검 구간을 기록했음.
AGB 단위는 지상부 Mg/ha (= t/ha)이며 탄소가 아님. 기준은 좌표 없는 나무도 포함했음.
6주차 계수를 그대로 썼으며 GRSM은 D2, ORNL은 D1 평가 방향만 적용했음. 재학습·보정하지 않았음.
예시 지도는 부호 있는 조사구 평균 높이 편향의 최솟값·최댓값 조사구를 선택했음.

## 관측일 및 단위 확인

"""
report += plot_features[['plotID', 'siteID', 'meta_observation_date', 'meta_unit_warning',
    'neon_nodata_frac', 'meta_nodata_frac', 'neon_nodata_warning_gt10pct', 'meta_nodata_warning_gt10pct']].to_markdown(index=False) + '\n\n'
report += '원본 CRS·픽셀 간격·dtype·nodata·tags·unit·읽은 window는 meta_sources.json에 기록했음. '
report += '높이 단위가 원본 tags에서 확인되지 않으면 경고를 남기고 m로 가정했으며 단위 변환은 하지 않았음. '
report += '파일 생성·수정일은 관측일로 쓰지 않았음. 관측일 미확인은 영상 시점이 일치함을 뜻하지 않음.\n\n'
report += f'실제 TIFF 전송량: {TRANSFER["bytes"] / 2**20:.2f} MiB (반복 Range 포함, 선택적 JSON/XML 제외)였음.\n\n'
report += '## 6주차 고정 계수\n\n' + coefficients.to_markdown(index=False, floatfmt='.8f') + '\n\n'
report += f'원본 coefficients.csv SHA-256: `{COEFFICIENTS_ORIGINAL_SHA256}`\n\n'
report += f'원본 features.csv SHA-256: `{FEATURES_ORIGINAL_SHA256}`\n\n'
report += f'기준 CSV: `{REFERENCE_PATH}`; 경계: `{FOOTPRINT_PATH}`; 실행 시각 UTC: {datetime.now(timezone.utc).isoformat()}\n\n'
report += """## 확인한 내용과 남은 한계

- 시점 차이: NEON LiDAR는 2021년, Meta 영상은 관측일 기록값(없으면 미확인, 명목상 2018–2020 전후). 차이에 실제 생장·교란이 섞일 수 있음
- 학습 자료 겹침: Meta 모델은 NEON 라이다로 학습했으므로 GRSM·ORNL이 학습에 쓰였을 수 있음. 그러면 결과가 실제보다 좋게 나옴. 국내(금정산) 정확도는 이보다 나쁠 수 있음
- 23개 조사구의 내부 비교이며 독립 검증 아님. 미국 동부 활엽수림 결과라 국내 숲에 그대로 적용하지 않음
- bilinear 재투영의 평활화·위치 정합 오차와 원자료의 시점·센서 차이가 함께 섞임. 이번 비교만으로 순수한 모델 오차를 분리할 수 없음
- 픽셀은 공간적으로 상관되어 있어 유효 픽셀 수를 독립 표본 수로 해석하지 않음
- 6주차 계수도 8~15개 학습 표본과 2개 지역에 한정됨. AGB 영향은 고정 모델의 입력 민감도 비교이며 Meta로 새 모델을 학습한 성능이 아님
- 평균비가 1에 가까워도 조사구별 오차·RMSE·편향·상관을 함께 봐야 함
- Meta 높이 단위·관측일이 미확인이면 해당 가정이 남음. 원본 tags와 관측일 기록을 먼저 확인해야 함

## 다음 단계 판단 (직접 채움)

- 국내 적용 판단: ____________________
- 근거 및 추가 검증: ____________________
"""
(WORK / 'report.md').write_text(report, encoding='utf-8')
expected = [*tables, 'meta_sources.json', 'report.md',
    *['figures/' + name for name in ['scatter_plot_mean_height.png', 'pixel_hexbin.png',
       'bias_by_height_bin.png', 'agb_neon_vs_meta.png', 'examples.png']]]
assert all((WORK / name).is_file() for name in expected)
assert len(pd.read_csv(WORK / 'plot_features.csv')) == 23
with zipfile.ZipFile(ZIP_PATH, mode='x', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in expected:
        archive.write(WORK / name, arcname=name)
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert set(archive.namelist()) == set(expected) and archive.testzip() is None
DRIVE_OUT.mkdir(parents=True, exist_ok=False)
for name in expected:
    target = DRIVE_OUT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    with (WORK / name).open('rb') as source, target.open('xb') as dest:
        shutil.copyfileobj(source, dest)
with ZIP_PATH.open('rb') as source, (DRIVE_OUT / ZIP_PATH.name).open('xb') as dest:
    shutil.copyfileobj(source, dest)
def sha_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
assert sha_file(ZIP_PATH) == sha_file(DRIVE_OUT / ZIP_PATH.name)
print('저장했음:', DRIVE_OUT, '| ZIP:', ZIP_PATH)
print('경과 시간(분):', round((time.monotonic() - START_TIME) / 60, 1))
files.download(str(ZIP_PATH))
display(Markdown('**조사구 평균 높이 비교 — Meta−NEON**'))
display(plot_metrics.loc[plot_metrics.feature == 'mean_height_m'])
display(Markdown('**고정 계수 AGB 비교 — NEON 입력 vs Meta 입력 (Mg/ha)**'))
display(agb_comparison)
